# Secondary PLL: 3.5 GHz, ±320-turn acquisition

ADC225/1 → DAC229/3. Use the matched IS03/PLL640 bit/HWH. Initialize board clocks
and load the overlay using your normal startup procedure first. Loading a bitstream
interrupts every loop; the import cell below does not download or configure it.
Run cells individually. Analog piezo/EOM controls remain external.


In [ ]:
import importlib
import laser_pll
laser_pll = importlib.reload(laser_pll)
# If attaching in a fresh notebook to an already running FPGA:
# from pynq import Overlay
# ol = Overlay('/path/to/TransportPhaseLockFF.bit', download=False)
target = laser_pll.channel(ol, 1)
print('Helper:', laser_pll.__file__)
print('Stage-3 full span [turns]:', laser_pll.stage3_phase_divisor(target))
assert laser_pll.stage3_phase_divisor(target) == 640


## Configure / acquire

The next cell mutes and resets this PLL for a fresh acquisition. Do not run it on
an existing lock. Restore the board's normal RFDC clock/startup settings before
using it. Match detector polarity to the analog path as before. The low-frequency
primary PLL and intensity controller are not configured by this cell.


In [ ]:
reference_hz = laser_pll.configure(
    target, reference_frequency_hz=3.5e9, phase_offset_rad=0,
    phase_gain_shift=0, capture_gain_shift=6, minimum_amplitude=64, invert=True)
laser_pll.configure_acquisition(target, stage=1, coarse_gain_shift=0,
    transition_interval_log2=2, minimum_fft_peak=32)
print('Reference [GHz]:', reference_hz/1e9)
# Enable when the analog acquisition paths are ready:
laser_pll.enable(target)
print(laser_pll.acquisition_status(target))


Move to stage 2 after coarse capture. The handoff still waits for its existing
valid-phase / near-frequency qualification. Inspect the scope and analog actuator
headroom; a successful software return is not proof of phase lock.


In [ ]:
laser_pll.select_stage(target, 2)
print(laser_pll.acquisition_status(target))


Enter stage 3 with the analog fast loop prepared as in the prior procedure.
Its signed 24-bit turn memory can wind up well beyond the DAC rails; avoid leaving
it integrating an uncontrolled detuning while waiting to enable the EOM.


In [ ]:
laser_pll.select_stage(target, 3)
print(laser_pll.acquisition_status(target))


## Tracking, after independently confirming phase lock

Start at the same /640 slope. This saves the phase/DAC operating point. Later
sensitivity changes ramp smoothly but still change loop gain. Increase one step
at a time and check for oscillation; lowering shift again is supported.


In [ ]:
laser_pll.set_tracking_gain(target, 0, wait=False)
laser_pll.select_stage(target, 4)
print(laser_pll.tracking_status(target))


In [ ]:
laser_pll.set_tracking_gain(target, 1)  # /320; next try 2 for /160, if stable.
print(laser_pll.tracking_status(target))


## I/Q monitor

This records baseband I/Q without reconfiguring the reference or gain. Decimation
averages samples, so choose it for the intended offset-frequency band and avoid
interpreting unresolved phase rotation as amplitude dropout.


In [ ]:
iq = laser_pll.capture_iq(target, samples=262144, decimation_log2=4)
# Use the same NPZ/PSD saving cells as your normal I/Q notebook.
